# 1. Prepare recordings for training

Run this notebook first, from top to bottom with **Shift+Enter**. Each numbered section explains its input and expected output. Change the **parameters cell** to choose participants, sensors, windows and forecast timing.

**Data flow:** `data/raw/sub-*` → filtered signal windows → labels → numerical features → saved handoff for notebook 2.

A **participant** is one person. A **recording/run** is one recording period for that person. A **window** is a short slice of that recording. A **feature** is a number describing the window, such as signal power. One eligible window becomes one row for training.

This is the main workflow; you do not need the dashboard or CLI scripts. The tested signal/label algorithms remain in `src/` so these notebooks stay readable. Run them inside this project rather than copying them out on their own.

**First-time setup (PowerShell, from the project root):**
```powershell
.venv/Scripts/python.exe -m pip install -r requirements.txt
.venv/Scripts/python.exe -m ipykernel install --user --name seizure-prediction --display-name "Python (seizure_prediction)"
.venv/Scripts/python.exe -m notebook .
```
Select **Python (seizure_prediction)** in both notebooks. VS Code can also use `.venv/Scripts/python.exe` directly.

Saved notebook outputs are intentionally empty. Running a cell produces its tables, plots and messages on your machine.


## 1. Load Python tools

**Input:** the project folder and its installed dependencies. **Output:** the project path and selected Python interpreter. No recordings are loaded yet.


In [ ]:
import json
import re
import sys
import time
import traceback
from contextlib import contextmanager, redirect_stdout, redirect_stderr
from datetime import datetime, timezone
from pathlib import Path

# Works when Jupyter starts in the project root or a subfolder of this project.
ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents]
             if (p / "src" / "utils" / "config.py").exists()), None)
if ROOT is None:
    raise RuntimeError("Open this notebook from inside the seizure_prediction project.")
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import yaml
from IPython import get_ipython
from IPython.display import display
from src.utils.config import load_config
from src.preprocessing.stream_processor import digest, save_json

if get_ipython() is not None:
    get_ipython().run_line_magic("matplotlib", "inline")
pd.set_option("display.max_columns", 12)
print("Project:", ROOT)
print("Python:", sys.executable)
print("Tip: if imports fail, select the project's .venv kernel.")


## 2. Choose the preprocessing parameters

**Input:** values below. **Output:** variables used by later cells. This cell does not process data.

`SUBJECTS = None` discovers all participants automatically: today's ten folders or a future cohort of 100+. Use a list such as `["sub-001", "sub-002"]` for a first look. Such a small cohort may prepare successfully but lack enough patients/events for training.

`RECORDINGS = None` selects all recordings for those people. For a smaller trial, use exact IDs found in section 3, for example `["sub-001_ses-01_task-szMonitoring_run-03"]`. A single recording is useful for debugging, not a train/validation/test study.

The teaching defaults keep the original **30–40 minute forecast**, five-second windows, filters and event exclusions. **The stride is 60 seconds instead of the original 5 seconds** to reduce work. Native sampling rates are retained instead of resampling everything to 256 Hz; tabular features support this. Sparse windows do not represent continuous monitoring. Set `STRIDE_SEC = 5` and `OUTPUT_SAMPLING_RATE = 256` for the original waveform settings.

At decision time `t`, label `1` means seizure onset in `[t + 30 min, t + 40 min)`. Label `0` is eligible background; `-1` is excluded. The model never receives the future signal or event annotations as input.


In [ ]:
DATASET_NAME = "seizeit2_example"   # Use the same name in notebook 2.
RAW_DIR = ROOT / "data" / "raw"
SUBJECTS = None                     # None = discover every participant; no fixed folder count.
RECORDINGS = None                   # None = every run; otherwise exact full recording IDs.
MODALITIES = ["eeg", "ecg", "emg", "mov"]
CHANNELS = {"eeg": None, "ecg": None, "emg": None, "mov": None}  # None = all channels.

WINDOW_SEC = 5                      # Observed signal length per example.
STRIDE_SEC = 60                     # Time between examples; larger = faster, fewer rows.
OUTPUT_SAMPLING_RATE = None         # None = native rates; 256 = a shared output rate.
FILTER_WARMUP_SEC = 10              # Additional past signal used to settle causal filters.
FILTERS = {
    "eeg": {"low_hz": 0.5, "high_hz": 40, "order": 4},
    "ecg": {"low_hz": 0.5, "high_hz": 40, "order": 4},
    "emg": {"low_hz": 10, "high_hz": 100, "order": 4},
    "mov": {"low_hz": None, "high_hz": None, "order": 4},
}
MINIMUM_WARNING_MIN = 30
OCCURRENCE_PERIOD_MIN = 10
POSTICTAL_EXCLUSION_MIN = 30         # Exclude signal after a seizure.
SEIZURE_PREFIXES = ["sz"]           # All seizure types; ["sz_foc"] = focal types only.
FEATURE_FAMILIES = {
    "eeg": ["time", "spectral", "hjorth", "entropy", "correlation"],
    "ecg": ["time", "spectral", "cardiac"],
    "emg": ["time", "spectral", "muscle"],
    "mov": ["time", "spectral", "motion"],
}
FEATURE_BATCH_WINDOWS = 128         # Number of windows processed together in feature extraction.
SKIP_BAD_RECORDINGS = False         # False = stop on errors; True = log explicit exclusions.
DEBUG = True                       # Show detailed exclusion and missing-value diagnostics.


## 3. Discover participants and check source files

**Input:** raw EDF recordings, JSON metadata and event TSV files. **Output:** a participant summary and a recording inventory. This reads filenames and small metadata files, not whole waveforms.

The seizure count here is the number of annotations, not the number of trainable events. Events too near the start/end of a run, or near another seizure, may have no eligible forecast windows. Missing files and unreadable event tables are shown explicitly.


In [ ]:
from src.data.bids_loader import BIDSLoader, require_complete_recordings

if not re.fullmatch(r"[A-Za-z0-9_-]+", DATASET_NAME):
    raise ValueError("DATASET_NAME must contain only letters, numbers, underscores or hyphens.")
if not Path(RAW_DIR).is_dir():
    raise FileNotFoundError(f"Raw folder not found: {RAW_DIR}")

selection = {"subjects": SUBJECTS, "recordings": RECORDINGS, "runs": None}
runs = BIDSLoader(RAW_DIR).selected_runs(selection)
if not runs:
    raise ValueError("No EDF recordings match these parameters. Check RAW_DIR and SUBJECTS.")
selected_subjects = sorted({run["subject"] for run in runs})
if SUBJECTS is not None and set(SUBJECTS) - set(selected_subjects):
    raise ValueError(f"Participants with no matching recordings: {set(SUBJECTS) - set(selected_subjects)}")

inventory_rows = []
for run in runs:
    required = set(MODALITIES) | {"eeg", "events"}  # EEG supplies the annotation time origin.
    missing = sorted(mod for mod in required if not run["files"].get(mod))
    annotation_error, seizure_count = "", None
    if run["files"].get("events"):
        try:
            annotations = pd.read_csv(run["files"]["events"], sep="\t")
            seizure_count = int(annotations["eventType"].str.startswith(tuple(SEIZURE_PREFIXES)).sum())
        except Exception as error:
            annotation_error = str(error)
    inventory_rows.append({"subject": run["subject"], "recording_id": run["id"],
                           "missing_files": ", ".join(missing),
                           "seizure_annotations": seizure_count, "annotation_error": annotation_error})
inventory = pd.DataFrame(inventory_rows)
display(inventory.groupby("subject").agg(recordings=("recording_id", "size"),
                                         seizure_annotations=("seizure_annotations", "sum")))
display(inventory.head(12))
print(f"Selected {len(selected_subjects)} participants and {len(runs)} recordings.")
problems = inventory.loc[inventory.missing_files.ne("") | inventory.annotation_error.ne("")]
if not problems.empty:
    display(problems)
    if not SKIP_BAD_RECORDINGS:
        raise ValueError("Source-file problems above. Fix the selection or explicitly enable SKIP_BAD_RECORDINGS.")


## 4. Save the exact preparation settings

**Input:** your parameters and discovered recording IDs. **Output:** a validated configuration file and a dedicated cache folder. The saved selection is frozen, so adding participant folders later does not change this snapshot.

The base YAML supplies advanced settings such as feature frequency bands and ECG beat detection. The values you chose above override it. The output prints the active settings; you can inspect the complete saved YAML if needed.

This reuses the project's published run-relative alignment assumption: the sensor streams share the publisher's run origin. EDF absolute dates are not used to align them. No continuity is assumed between runs. New datasets need their own timing and annotation-coverage checks.


In [ ]:
config = load_config(ROOT / "src" / "config" / "config.yaml")
config["dataset"].update(raw_dir=str(Path(RAW_DIR).resolve()), subjects=selected_subjects,
                         runs=None, recordings=sorted(run["id"] for run in runs))
config["sensors"].update(enabled=MODALITIES, channels=CHANNELS, missing_policy="error")
config["preprocessing"].update(window_sec=WINDOW_SEC, stride_sec=STRIDE_SEC, context_sec=0,
                                output_sampling_rate=OUTPUT_SAMPLING_RATE,
                                filter_warmup_sec=FILTER_WARMUP_SEC, filters=FILTERS)
config["labels"].update(enabled=True, minimum_warning_min=MINIMUM_WARNING_MIN,
                        occurrence_period_min=OCCURRENCE_PERIOD_MIN,
                        postictal_exclusion_min=POSTICTAL_EXCLUSION_MIN, seizure_prefixes=SEIZURE_PREFIXES)
config["features"].update(enabled=True, families=FEATURE_FAMILIES, batch_windows=FEATURE_BATCH_WINDOWS)
config["qualification"].update(enabled=SKIP_BAD_RECORDINGS, exclude_unassigned_subjects=False,
                               continue_on_recording_error=SKIP_BAD_RECORDINGS)

# The folder hash separates different sensor/window/label/feature selections.
identity = {key: config[key] for key in ("dataset", "sensors", "alignment", "preprocessing", "labels", "features", "qualification")}
identity["dataset"] = {k: v for k, v in config["dataset"].items() if k != "interim_dir"}
WORKSPACE = ROOT / "audit" / "runs" / "notebooks" / DATASET_NAME
PREPARED_DIR = WORKSPACE / "prepared" / digest(identity)
PREPARED_DIR.mkdir(parents=True, exist_ok=True)
config["dataset"]["interim_dir"] = str(PREPARED_DIR)
CONFIG_FILE = PREPARED_DIR / "config.yaml"
CONFIG_FILE.write_text(yaml.safe_dump(config, sort_keys=False), encoding="utf-8")
config = load_config(CONFIG_FILE)  # Uses the same parameter validation as the CLI.
HANDOFF_FILE = WORKSPACE / "handoff.json"
LOG_FILE = PREPARED_DIR / "preparation.log"
print("Configuration:", CONFIG_FILE)
display(pd.Series({"participants": len(selected_subjects), "recordings": len(runs),
                   "sensors": MODALITIES, "window_seconds": WINDOW_SEC, "stride_seconds": STRIDE_SEC,
                   "output_rate": OUTPUT_SAMPLING_RATE or "native per sensor",
                   "forecast_minutes": [MINIMUM_WARNING_MIN, MINIMUM_WARNING_MIN + OCCURRENCE_PERIOD_MIN],
                   "skip_bad_recordings": SKIP_BAD_RECORDINGS}, name="active settings"))


## 5. Enable progress logging and estimate disk space

**Input:** selected recording headers and the active settings. **Output:** an approximate signal-cache size, free disk space and a log path. Headers are read without loading whole recordings into RAM. The estimate excludes feature tables, metadata and old caches.

Preprocessing writes arrays to disk in bounded windows. Training later loads a feature matrix into RAM and has a separate memory limit. More participants are discovered automatically, but a much larger cohort may need a smaller experiment or more RAM/storage.


In [ ]:
# Tee sends ordinary printed output to both the notebook and a text log.
class Tee:
    def __init__(self, screen, log):
        self.screen, self.log = screen, log

    def write(self, text):
        self.screen.write(text)
        self.log.write(text)
        return len(text)

    def flush(self):
        self.screen.flush()
        self.log.flush()


@contextmanager
def logged_step(name):
    start = time.perf_counter()
    with LOG_FILE.open("a", encoding="utf-8") as log:
        with redirect_stdout(Tee(sys.stdout, log)), redirect_stderr(Tee(sys.stderr, log)):
            print(f"\nSTART {name} | {datetime.now(timezone.utc).isoformat()}", flush=True)
            try:
                yield
            except Exception:
                traceback.print_exc()  # Keep the full error in the log, then stop the cell.
                raise
            finally:
                print(f"END {name} | elapsed {time.perf_counter() - start:.1f} seconds", flush=True)


print("Detailed progress and errors:", LOG_FILE)


import shutil
import mne

estimated_bytes = 0
for run in runs:
    headers = []
    for mod in MODALITIES:
        path = run["files"].get(mod)
        if path is None:
            continue
        raw = mne.io.read_raw_edf(path, preload=False, verbose=False)
        try:
            if CHANNELS.get(mod) is not None:
                raw.pick(CHANNELS[mod])
            headers.append((raw.n_times / raw.info["sfreq"], len(raw.ch_names), raw.info["sfreq"]))
        finally:
            raw.close()
    if headers:
        duration = min(h[0] for h in headers)
        windows = max(0, int((duration - FILTER_WARMUP_SEC - WINDOW_SEC) // STRIDE_SEC) + 1)
        estimated_bytes += windows * sum(channels * round(WINDOW_SEC * (OUTPUT_SAMPLING_RATE or rate)) * 4
                                         for _, channels, rate in headers)
free_bytes = shutil.disk_usage(PREPARED_DIR).free
print(f"Approximate new signal arrays: {estimated_bytes / 1024**3:.2f} GiB")
print(f"Free disk space: {free_bytes / 1024**3:.2f} GiB")
if estimated_bytes > free_bytes * 0.8:
    raise RuntimeError("Too little disk headroom. Select fewer recordings or increase STRIDE_SEC.")


## 6. Turn recordings into filtered windows and labels

**Input:** raw signals plus annotations. **Output:** memory-mapped `.npy` waveform arrays, a `windows.csv` per run and `latest_run.json` listing their locations. Each array has shape **windows × channels × samples**.

This cell filters each electrical sensor, checks finite/flat signals, uses only the common sensor interval, and assigns forecast labels. Additional past filter warm-up is included in exclusion checks. It does not fit a model, fill feature missing values or learn scaling.

One progress line appears per recording. Interrupting the kernel preserves completed caches. Rerun from the top with the same parameters to reuse them. Avoid running two preparation notebooks against the same output folder simultaneously. A new handoff is written only after section 10 succeeds.


In [ ]:
from src.pipeline.preparation import prepare_recordings

# Invalidate the previous handoff while this dataset name is being rebuilt.
HANDOFF_FILE.unlink(missing_ok=True)
with logged_step("Prepare signal windows and forecast labels"):
    preprocessing_reports = prepare_recordings(config)

preprocessing_summary = pd.DataFrame([
    {key: item.get(key) for key in ("run", "status", "windows", "eligible", "signal_cache_reused", "error")}
    for item in preprocessing_reports
])
display(preprocessing_summary)
print("eligible here is BEFORE any extra feature-history exclusions.")


## 7. Inspect one prepared waveform and its label

**Input:** the first completed recording. **Output:** one window's metadata, per-sensor shapes and a plot of one channel per sensor. This is a visual debugging example, not proof that every recording is clean. Electrical values are calibrated volts; motion retains its EDF-calibrated units.

Different native sampling rates mean different sample counts for the same five-second interval. The feature step supports this. The first window can be excluded; section 9 reports the final trainable rows.


In [ ]:
first_report = next(item for item in preprocessing_reports if item["status"] == "complete")
window_table = pd.read_csv(Path(first_report["result_directory"]) / "windows.csv")
display(window_table.head())
fig, axes = plt.subplots(len(first_report["inventory"]), 1, figsize=(11, 2.2 * len(first_report["inventory"])), squeeze=False)
for ax, (mod, info) in zip(axes[:, 0], first_report["inventory"].items()):
    array = np.load(Path(first_report["signal_directory"]) / f"{mod}.npy", mmap_mode="r")
    try:
        waveform = np.array(array[0, 0], copy=True)  # Copy only one channel of one window.
        print(mod, "array shape:", array.shape, "rate:", info["output_rate"], "Hz")
        ax.plot(np.arange(len(waveform)) / info["output_rate"], waveform)
        ax.set(title=f"{mod}: {info['channels'][0]}", xlabel="Seconds within window", ylabel="Calibrated value")
    finally:
        array._mmap.close()
fig.tight_layout()
plt.show()


## 8. Convert signal windows into a trainable feature table

**Input:** prepared waveform windows. **Output:** one feature table per run, a feature dictionary explaining column names, updated eligibility, and `feature_manifest.json` linking all files.

Examples: time features describe amplitude/variability; spectral features describe frequency power; cardiac features describe beats and short-window RR variability; motion features describe movement. ECG cardiac features need 60 seconds of preceding observed history. Their eligibility is checked here, so the final row count can decrease.

Columns look like `eeg::channel::feature`. The table contains signal-derived numbers; identifiers, times and labels remain separate. Learned missing-value filling, scaling and feature selection belong inside training folds in notebook 2.


In [ ]:
from src.features.build import build_features

with logged_step("Extract numerical signal features"):
    feature_entries = build_features(config)
print("Feature manifest:", PREPARED_DIR / "feature_manifest.json")
print("Recordings with eligible feature rows:", len(feature_entries))


## 9. Check the final rows and investigate exclusions

**Input:** saved feature/window tables. **Output:** per-participant background/positive counts, exclusion reasons, a small feature preview and (when `DEBUG=True`) missing-value diagnostics.

**Positive windows are not independent seizures.** Many windows may refer to one event. Notebook 2 counts unique events and checks both classes in every partition/fold. If there are no positives, inspect forecast timing, run duration, exclusions and annotation types before changing model settings.


In [ ]:
feature_summary = pd.DataFrame([
    {"subject": e["subject"], "recording_id": e["run"], "all_windows": e["rows"],
     "eligible": e["eligible"], "positive": e["positive"], "features": e["features"],
     "missing_fraction": e["missing_fraction"]} for e in feature_entries
])
feature_summary["background"] = feature_summary.eligible - feature_summary.positive
display(feature_summary.groupby("subject")[["all_windows", "eligible", "background", "positive"]].sum())
missing_subjects = set(selected_subjects) - set(feature_summary.subject)
print("Participants without eligible feature rows:", sorted(missing_subjects) or "none")

example = feature_entries[0]
example_features = pd.read_pickle(example["feature_file"])  # Only load artifacts generated locally by this project.
example_windows = pd.read_csv(example["windows_file"])
display(example_features.loc[example_windows.eligible].iloc[:5, :8])
dictionary = json.loads((Path(example["feature_file"]).parent / "feature_dictionary.json").read_text())
display(pd.DataFrame.from_dict(dictionary, orient="index").head(8))

if DEBUG:
    reasons = pd.Series(dtype="int64")
    for entry in feature_entries:
        reasons = reasons.add(pd.Series(entry["reason_counts"], dtype="int64"), fill_value=0)
    display(reasons.sort_values(ascending=False).rename("window reason counts in feature manifest"))
    display(example_features.isna().mean().sort_values(ascending=False).head(12).rename("missing fraction in example run"))
    qualification_file = PREPARED_DIR / "qualification.json"
    if qualification_file.exists():
        display(pd.DataFrame(json.loads(qualification_file.read_text()).get("excluded", [])))

if feature_summary.positive.sum() == 0:
    raise ValueError("No positive forecast windows. Review timing and exclusions above; no training handoff was created.")


## 10. Publish the handoff to notebook 2

**Input:** successful prepared files. **Output:** `audit/runs/notebooks/<DATASET_NAME>/handoff.json`, pointing to this exact snapshot and its settings. Notebook 2 reads this file, so you do not need to copy paths or re-enter preprocessing parameters.

The handoff means preparation completed; training readiness is checked in notebook 2. You may now close this kernel. Future model experiments reuse the same handoff. To include new participants or change signals/labels/features, rerun this notebook; keep the same dataset name to update its handoff, or choose a new name to keep experiments separate.


In [ ]:
feature_summary.to_csv(PREPARED_DIR / "dataset_summary.csv", index=False)
handoff = {
    "schema_version": 1, "status": "prepared",
    "created_utc": datetime.now(timezone.utc).isoformat(),
    "config_file": str(CONFIG_FILE),
    "config_digest": digest(config),
    "feature_manifest": str(PREPARED_DIR / "feature_manifest.json"),
    "manifest_digest": digest(feature_entries),
    "subjects": sorted(feature_summary.subject.unique().tolist()),
    "selected_subjects": selected_subjects,
    "eligible_windows": int(feature_summary.eligible.sum()),
    "positive_windows": int(feature_summary.positive.sum()),
}
temporary_handoff = HANDOFF_FILE.with_suffix(".tmp")
save_json(temporary_handoff, handoff)
temporary_handoff.replace(HANDOFF_FILE)  # The pointer appears only when its content is complete.
print("PREPARATION COMPLETE")
print("Handoff:", HANDOFF_FILE)
print("Next: open 02_train_model.ipynb and use DATASET_NAME =", repr(DATASET_NAME))


## If a cell fails

Read the last error and `preparation.log`. Fix the parameter/input, then use **Kernel → Restart Kernel and Run All Cells**. This avoids stale variables after changing parameters. Completed signal caches can be reused; changed signal settings create a different cache.

| Symptom | What to inspect |
|---|---|
| Import error | The selected Python interpreter and `requirements.txt` |
| No recordings | `RAW_DIR`, participant names and exact recording IDs |
| Missing channel/modality | The inventory and exact EDF channel names; do not invent missing signals |
| No eligible/positive rows | Forecast interval, per-run coverage, event types and exclusion reasons |
| Duration/date/rate mismatch | EDF/JSON metadata and the documented alignment assumption |
| Too much disk/time | Fewer participants/recordings, fewer modalities, or a larger stride |

For interactive debugging, add a temporary code cell containing `%debug` immediately after an error. Use `p variable_name` to inspect a variable and `q` to exit. You can also temporarily add `breakpoint()` before a failing operation. Remove debugging stops before running all cells.
